## Chapter 5-01: Feature Engineering on Databricks/Project:
* Streaming Transactions/CH5-01-Generating Records.py

In [0]:
# codes from Chapter 3: Building Our Bronze Layer/Project: Streaming Transactions/CH3-01-Generating Records and Schema Change.py

dbutils.widgets.dropdown(name='Reset', defaultValue='True', choices=['True', 'False'], label="Reset: Delete previous data")
# Create a Unity Catalog Volume
spark.sql("CREATE VOLUME IF NOT EXISTS porya_catalog.default.synthetic_transactions")
# Sets the file path variable 
volume_file_path = "/Volumes/porya_catalog/default/synthetic_transactions/"

In [0]:
%pip install -q dbldatagen

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC Chapter 5: Feature Engineering in Unity Catalog
# MAGIC
# MAGIC ## Transaction data - Record Generator
# MAGIC We simulate streaming data by generating labeled JSON data. Then we write it to a folder in our volume.



# DBTITLE 1,Notebook Variables
nRows = 10
nPositiveRows = round(nRows/3)
destination_path = "{}/raw_transactions/data".format(volume_file_path)  #this changed
temp_path = "{}/temp".format(volume_file_path)
sleepIntervalSeconds = 1

# COMMAND ----------

if bool(dbutils.widgets.get('Reset')):
  dbutils.fs.rm(temp_path, recurse=True)
  dbutils.fs.rm(destination_path, recurse=True)

dbutils.fs.mkdirs(destination_path)

# COMMAND ----------

# DBTITLE 1,Data Variables
CustomerID_vars = {"min": 1234, "max": 1260}

Product_vars = {"A": {"min": 1000, "max": 25001, "mean": 15520, "alpha": 4, "beta": 10},
                "B": {"min": 1000, "max": 5501, "mean": 35520, "alpha": 10, "beta": 4},
                "C": {"min": 10000, "max": 40001, "mean": 30520, "alpha": 3, "beta": 10}}  # this changed

# COMMAND ----------

import dbldatagen as dg
from datetime import datetime
import dbldatagen.distributions as dist
from pyspark.sql.types import IntegerType, FloatType, StringType

def define_specs(Product, Label, currentTimestamp = datetime.now()):
  pVars = Product_vars[Product]
  if Product == "None":
    if Label:
      return (dg.DataGenerator(spark, name="syn_trans", rows=nRows)
        .withColumn("CustomerID", IntegerType(), nullable=False,
                    minValue=CustomerID_vars["min"], maxValue=CustomerID_vars["max"], random=True)
        .withColumn("TransactionTimestamp", "timestamp", 
                    begin=currentTimestamp, end=currentTimestamp,nullable=False,
                  random=False)
        .withColumn("Amount", FloatType(), 
                    minValue=pVars["min"],maxValue=pVars["max"], 
                    distribution=dist.Beta(alpha=pVars["alpha"], beta=pVars["beta"]), random=True)
        .withColumn("Label", IntegerType(), minValue=1, maxValue=1)).build()
    else:
      return (dg.DataGenerator(spark, name="syn_transs", rows=nRows, partitions=4)
        .withColumn("CustomerID", IntegerType(), nullable=False,
                    minValue=CustomerID_vars["min"], maxValue=CustomerID_vars["max"], random=True)
        .withColumn("TransactionTimestamp", "timestamp", 
                    begin=currentTimestamp, end=currentTimestamp,nullable=False,
                  random=False)
        .withColumn("Amount", FloatType(), 
                    minValue=pVars["min"],maxValue=pVars["max"], 
                    distribution=dist.Normal(mean=pVars["mean"], stddev=.001), random=True)
        .withColumn("Label", IntegerType(), minValue=0, maxValue=0)).build()
  else:
    if Label:
      return (dg.DataGenerator(spark, name="syn_trans", rows=nRows, partitions=4)
        .withColumn("CustomerID", IntegerType(), nullable=False,
                    minValue=CustomerID_vars["min"], maxValue=CustomerID_vars["max"], random=True)
        .withColumn("TransactionTimestamp", "timestamp", 
                    begin=currentTimestamp, end=currentTimestamp,nullable=False,
                  random=False)
        .withColumn("Product", StringType(), template=f"Pro\duct \{Product}") 
        .withColumn("Amount", FloatType(), 
                    minValue=pVars["min"],maxValue=pVars["max"], 
                    distribution=dist.Beta(alpha=pVars["alpha"], beta=pVars["beta"]), random=True)
        .withColumn("Label", IntegerType(), minValue=1, maxValue=1)).build()
    else:
      return (dg.DataGenerator(spark, name="syn_transs", rows=nRows, partitions=4)
        .withColumn("CustomerID", IntegerType(), nullable=False,
                    minValue=CustomerID_vars["min"], maxValue=CustomerID_vars["max"], random=True)
        .withColumn("TransactionTimestamp", "timestamp", 
                    begin=currentTimestamp, end=currentTimestamp,nullable=False,
                  random=False)
        .withColumn("Product", StringType(), template=f"Pro\duct \{Product}")
        .withColumn("Amount", FloatType(), 
                    minValue=pVars["min"],maxValue=pVars["max"], 
                    distribution=dist.Normal(mean=pVars["mean"], stddev=.001), random=True)
        .withColumn("Label", IntegerType(), minValue=0, maxValue=0)).build()

# COMMAND ----------

display(define_specs(Product="B", Label=1, currentTimestamp=datetime.now()))


# COMMAND ----------

# DBTITLE 1,Functions to generate a JSON dataset for Auto Loader to pick up
from pyspark.sql.functions import expr
from functools import reduce
import pyspark
import os

# Generate a record
def generateRecord(Product,Label):
  return (define_specs(Product=Product, Label=Label, currentTimestamp=datetime.now()))
  
# Generate a list of records
def generateRecordSet(Products):
  Labels = [0,1]
  recordSet = []
  for Prod in Products:
    for Lab in Labels:
      recordSet.append(generateRecord(Prod, Lab))
  return reduce(pyspark.sql.dataframe.DataFrame.unionByName, recordSet)


# Generate a set of data, convert it to a Dataframe, write it out as one json file to the temp path. Then move that file to the destination_path
def writeJsonFile(destination_path, Products = list(Product_vars.keys())):
  recordDF = generateRecordSet(Products)
  recordDF = recordDF.withColumn("Amount", expr("Amount / 100"))
  recordDF.coalesce(1).write.format("json").save(temp_path)
  
  # Grab the file from the temp location, write it to the location we want and then delete the temp directory
  tempJson = os.path.join(temp_path, dbutils.fs.ls(temp_path)[3][1])
  dbutils.fs.cp(tempJson, destination_path)
  dbutils.fs.rm(temp_path, True)

# COMMAND ----------

# DBTITLE 1,Loop for Generating Data
import time

Products = list(Product_vars.keys())
t=0
total = 100
while(t<total):
  writeJsonFile(destination_path,Products=Products)
  if not (t % 10):
    print(f"t={t}")
  t = t+1
  time.sleep(sleepIntervalSeconds)

## Chapter 5-02: Feature Engineering on Databricks/Project:
* Streaming Transactions/CH5-02-Auto Loader.py

In [0]:
# DBTITLE 1,Variables
table_name = "porya_catalog.default.raw_transactions"
raw_data_location = f"{volume_file_path}/raw_transactions/data/"  # this changed
schema_location = f"{volume_file_path}/raw_transactions/schema"
checkpoint_location = f"{volume_file_path}/raw_transactions/checkpoint"

# COMMAND ----------

# DBTITLE 1,Use to reset for fresh table, schema, checkpoints
if bool(dbutils.widgets.get('Reset')):
  dbutils.fs.rm(schema_location, True)
  dbutils.fs.rm(checkpoint_location, True)
  sql(f"DROP TABLE IF EXISTS {table_name}")

# COMMAND ----------

# DBTITLE 1,Optimization settings and reduce the number of files that must be read to determine schema
for cfg, val in [
    ("spark.databricks.delta.optimizeWrite.enabled", True),
    ("spark.databricks.delta.autoCompact.enabled", True),
    ("spark.databricks.cloudFiles.schemaInference.sampleSize.numFiles", 1),
    ("spark.databricks.delta.schema.autoMerge.enabled", True),
]:
    try:
        spark.conf.set(cfg, val)
    except Exception as e:
        print(f"Skipping config {cfg}: {e}")

# COMMAND ----------

# DBTITLE 1,Readstream
stream = spark.readStream \
  .format("cloudFiles") \
  .option("cloudFiles.format", "json") \
  .option("cloudFiles.schemaHints","CustomerID int, Amount float, TransactionTimestamp timestamp, Product string") \
  .option("cloudFiles.inferColumnTypes","true") \
  .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
  .option("cloudFiles.schemaLocation", schema_location) \
  .load(raw_data_location) \
  .select("*") \
  .writeStream \
  .format("delta") \
  .outputMode("append") \
  .option("checkpointLocation", checkpoint_location) \
  .option("mergeSchema", "true") \
  .trigger(availableNow=True) \
  .toTable(tableName=table_name)

##Chapter 5-03: Feature Engineering on Databricks/Project:
* Streaming Transactions/CH5-03-FE Using Spark Structured Streaming.scala

In [0]:
# Databricks notebook source
from delta.tables import DeltaTable as dt
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructType, StructField, LongType, DoubleType, TimestampType, IntegerType

class InputRow:
    def __init__(self, CustomerID, TransactionTimestamp):
        self.CustomerID = CustomerID
        self.TransactionTimestamp = TransactionTimestamp

class TransactionCountState:
    def __init__(self, latestTimestamp, currentTransactions):
        self.latestTimestamp = latestTimestamp
        self.currentTransactions = currentTransactions

class TransactionCount:
    def __init__(self, CustomerID, transactionCount, eventTimestamp, isTimeout):
        self.CustomerID = CustomerID
        self.transactionCount = transactionCount
        self.eventTimestamp = eventTimestamp
        self.isTimeout = isTimeout

def add_new_records(new_records, transaction_count_state):
    record_with_latest_timestamp = max(new_records, key=lambda record: record.TransactionTimestamp)
    latest_new_timestamp = record_with_latest_timestamp.TransactionTimestamp

    latest_timestamp = latest_new_timestamp if latest_new_timestamp > transaction_count_state.latestTimestamp else transaction_count_state.latestTimestamp

    return TransactionCountState(latest_timestamp, transaction_count_state.currentTransactions + new_records)

def drop_expired_records(new_latest_timestamp, current_transactions):
    new_transaction_list = []
    expiration_timestamp = new_latest_timestamp - F.interval(2, "minutes")

    for transaction in current_transactions:
        if transaction.TransactionTimestamp >= expiration_timestamp:
            new_transaction_list.append(transaction)

    return TransactionCountState(new_latest_timestamp, new_transaction_list)

def update_state(customer_id, values, state):
    transaction_counts = []

    if not state.hasTimedOut:
        transaction_list = []
        for value in values:
            transaction_list.append(value)

        prev_state = state.getOption() or TransactionCountState(transaction_list[0].TransactionTimestamp, [])

        state_with_new_records = add_new_records(transaction_list, prev_state)
        state_with_records_dropped = drop_expired_records(state_with_new_records.latestTimestamp, state_with_new_records.currentTransactions)

        # Check if it's time to emit a record for this customer
        if F.current_timestamp() >= state_with_records_dropped.latestTimestamp + F.interval(1, "minute"):
            output = TransactionCount(customer_id, len(state_with_records_dropped.currentTransactions), F.current_timestamp(), False)
            transaction_counts.append(output)

        state.update(state_with_records_dropped)
        state.setTimeoutTimestamp(state_with_records_dropped.latestTimestamp + F.interval(30, "seconds"))
    else:
        prev_state = state.get()
        new_timestamp = F.current_timestamp()
        
        # Check if it's time to emit a record for this customer
        if new_timestamp >= prev_state.latestTimestamp + F.interval(1, "minute"):
            state_with_records_dropped = drop_expired_records(new_timestamp, prev_state.currentTransactions)
            output = TransactionCount(customer_id, len(state_with_records_dropped.currentTransactions), new_timestamp, True)
            transaction_counts.append(output)

        state.update(state_with_records_dropped)
        state.setTimeoutTimestamp(state_with_records_dropped.latestTimestamp + F.interval(30, "seconds"))

    return transaction_counts

schema = StructType([
    StructField("TransactionTimestamp", StringType(), True),
    StructField("CustomerID", IntegerType(), True),
    StructField("Amount", DoubleType(), True),
    StructField("Product", StringType(), True)
])

input_df = spark.readStream \
    .format("delta") \
    .schema(schema) \
    .table(table_name) \
    .selectExpr("CustomerID", "cast(TransactionTimestamp as timestamp) TransactionTimestamp")

In [0]:

from datetime import datetime
import time
# Note: GroupState, GroupStateTimeout, OutputMode are Scala-only Spark APIs.
# In Python, use applyInPandasWithState for stateful streaming.

for cfg, val in [
    ("spark.databricks.delta.optimizeWrite.enabled", "true"),
    ("spark.databricks.delta.autoCompact.enabled", "true"),
]:
    try:
        spark.conf.set(cfg, val)
    except Exception as e:
        print(f"Skipping config {cfg}: {e}")

In [0]:
ft_table_name = "porya_catalog.default.transaction_count_ft"
history_table_name = "porya_catalog.default.transaction_count_history"
volume_path = "/Volumes/porya_catalog/default/synthetic_transactions"
output_path = f"{volume_path}/transaction_count_ft/streaming_outputs/"
output_path2 = f"{volume_path}/transaction_count_history/streaming_outputs/"
input_table = "porya_catalog.default.raw_transactions"

# maybe we need a try catch here
sql(f"""CREATE OR REPLACE TABLE {ft_table_name} (CustomerID Int, transactionCount Int, eventTimestamp Timestamp, isTimeout Boolean)""")
sql(f"""ALTER TABLE {ft_table_name} ALTER COLUMN CustomerID SET NOT NULL""")
sql(f"""ALTER TABLE {ft_table_name} ADD PRIMARY KEY(CustomerID)""")
sql(f"ALTER TABLE {ft_table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed=true)")


sql(f"""CREATE OR REPLACE TABLE {history_table_name} (CustomerID Int, transactionCount Int, eventTimestamp Timestamp, isTimeout Boolean)""")
sql(f"""ALTER TABLE {history_table_name} ALTER COLUMN CustomerID SET NOT NULL""")
sql(f"""ALTER TABLE {history_table_name} ALTER COLUMN eventTimestamp SET NOT NULL""")
sql(f"""ALTER TABLE {history_table_name} ADD PRIMARY KEY(CustomerID, eventTimestamp TIMESERIES)""")

# aggregate transactions for windowMinutes
window_minutes = 2
# wait at most max_wait_minutes before writing a record
max_wait_minutes = 1

In [0]:
from datetime import datetime, timedelta
from typing import List

class InputRow:
    def __init__(self, CustomerID: int, TransactionTimestamp: datetime):
        self.CustomerID = CustomerID
        self.TransactionTimestamp = TransactionTimestamp

class TransactionCountState:
    def __init__(self, latestTimestamp: datetime, currentTransactions: List[InputRow]):
        self.latestTimestamp = latestTimestamp
        self.currentTransactions = currentTransactions

class TransactionCount:
    def __init__(self, CustomerID: int, transactionCount: int, eventTimestamp: datetime, isTimeout: bool):
        self.CustomerID = CustomerID
        self.transactionCount = transactionCount
        self.eventTimestamp = eventTimestamp
        self.isTimeout = isTimeout

def add_new_records(new_records: List[InputRow], transaction_count_state: TransactionCountState) -> TransactionCountState:
    record_with_latest_timestamp = max(new_records, key=lambda record: record.TransactionTimestamp)
    latest_new_timestamp = record_with_latest_timestamp.TransactionTimestamp
    latest_timestamp = max(latest_new_timestamp, transaction_count_state.latestTimestamp)
    return TransactionCountState(latest_timestamp, transaction_count_state.currentTransactions + new_records)

def drop_expired_records(new_latest_timestamp: datetime, current_transactions: List[InputRow], window_minutes: int) -> TransactionCountState:
    expiration_timestamp = new_latest_timestamp - timedelta(minutes=window_minutes)
    new_transaction_list = [t for t in current_transactions if t.TransactionTimestamp >= expiration_timestamp]
    return TransactionCountState(new_latest_timestamp, new_transaction_list)

def update_state(customer_id, values, state, window_minutes=2):
    transaction_counts = []
    values_list = list(values)
    now = datetime.now()

    if not state.hasTimedOut:
        transaction_list = values_list
        if state.exists:
            prev_state = state.get()
        else:
            first_transaction_timestamp = transaction_list[0].TransactionTimestamp if transaction_list else now
            prev_state = TransactionCountState(first_transaction_timestamp, [])
        state_with_new_records = add_new_records(transaction_list, prev_state)
        state_with_records_dropped = drop_expired_records(state_with_new_records.latestTimestamp, state_with_new_records.currentTransactions, window_minutes)
        output = TransactionCount(customer_id, len(state_with_records_dropped.currentTransactions), state_with_records_dropped.latestTimestamp, False)
        transaction_counts.append(output)
        state.update(state_with_records_dropped)
        state.setTimeoutDuration(30 * 1000)
    else:
        prev_state = state.get()
        new_timestamp = now
        state_with_records_dropped = drop_expired_records(new_timestamp, prev_state.currentTransactions, window_minutes)
        output = TransactionCount(customer_id, len(state_with_records_dropped.currentTransactions), state_with_records_dropped.latestTimestamp, True)
        transaction_counts.append(output)
        state.update(state_with_records_dropped)
        state.setTimeoutDuration(30 * 1000)
    return iter(transaction_counts)

In [0]:
# DBTITLE 1,Schema for checking type
from pyspark.sql.types import StringType, StructField, StructType, IntegerType, FloatType, TimestampType, BooleanType, ArrayType

# The schema for the incoming records
schema = StructType([
    StructField("Source", StringType(), True),
    StructField("TransactionTimestamp", StringType(), True),
    StructField("CustomerID", IntegerType(), True),
    StructField("Amount", FloatType(), True),
    StructField("Product", StringType(), True),
    StructField("Label", IntegerType(), True),
])

# COMMAND ----------

# DBTITLE 1,Read and write streams
from delta.tables import DeltaTable
from pyspark.sql.functions import expr
import pandas as pd
from datetime import datetime, timedelta

inputDf = (
    spark.readStream
    .format("delta")
    .schema(schema)
    .table(input_table)
    .selectExpr("CustomerID", "cast(TransactionTimestamp as timestamp) TransactionTimestamp")
)

# We're allowing data to be 30 seconds late before it is dropped
inputDf = inputDf.withWatermark("TransactionTimestamp", "30 seconds")

from pyspark.sql.streaming.state import GroupState, GroupStateTimeout

# State schema: track latest timestamp and count of current transactions
state_schema = StructType([
    StructField("latestTimestamp", TimestampType(), True),
    StructField("transactionCount", IntegerType(), True)
])

transaction_count_schema = StructType([
    StructField("CustomerID", IntegerType(), False),
    StructField("transactionCount", IntegerType(), False),
    StructField("eventTimestamp", TimestampType(), False),
    StructField("isTimeout", BooleanType(), False)
])

def update_state_py(key, pdf_iter, state):
    # Spark 4.1: 2nd param is Iterator[pd.DataFrame], consume it first
    pdf = pd.concat(list(pdf_iter))
    customer_id = key[0]
    now = datetime.now()
    window_td = timedelta(minutes=window_minutes)

    if state.hasTimedOut:
        if state.exists:
            prev = state.get
            latest_ts = prev[0]
            count = prev[1]
        else:
            latest_ts = now
            count = 0
        cutoff = now - window_td
        new_count = count if latest_ts and latest_ts >= cutoff else 0
        state.update((now, new_count))
        yield pd.DataFrame({
            "CustomerID": [customer_id],
            "transactionCount": [new_count],
            "eventTimestamp": [now],
            "isTimeout": [True]
        })
    else:
        new_count = len(pdf)
        if not pdf.empty:
            latest_ts = pdf["TransactionTimestamp"].max()
        else:
            latest_ts = now

        if state.exists:
            prev = state.get
            prev_latest = prev[0]
            prev_count = prev[1]
            cutoff = latest_ts - window_td
            if prev_latest and prev_latest >= cutoff:
                new_count += prev_count
            latest_ts = max(latest_ts, prev_latest) if prev_latest else latest_ts

        state.update((latest_ts, new_count))
        state.setTimeoutDuration(30 * 1000)
        yield pd.DataFrame({
            "CustomerID": [customer_id],
            "transactionCount": [new_count],
            "eventTimestamp": [latest_ts],
            "isTimeout": [False]
        })

flatMapGroupsWithStateResultDf = (
    inputDf.groupBy('CustomerID')
    .applyInPandasWithState(
        func=update_state_py,
        outputStructType=transaction_count_schema,
        stateStructType=state_schema,
        outputMode="append",
        timeoutConf="ProcessingTimeTimeout"
    )
)

def updateCounts(newCountsDf, epoch_id):
    aggregationTable = DeltaTable.forName(spark, ft_table_name)
    aggregationTable.alias("t") \
        .merge(
            newCountsDf.alias("m"),
            "m.CustomerID = t.CustomerID"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()

flatMapGroupsWithStateResultDf.writeStream \
    .foreachBatch(updateCounts) \
    .option("checkpointLocation", f"{output_path}/checkpoint") \
    .trigger(availableNow=True) \
    .queryName("flatMapGroups") \
    .start()

# COMMAND ----------

# DBTITLE 1,Writing the history of transaction counts to a table
flatMapGroupsWithStateResultDf.writeStream \
    .option("checkpointLocation", f"{output_path2}/checkpoint") \
    .trigger(availableNow=True) \
    .queryName("flatMapGroupsHistoryTable") \
    .toTable(history_table_name)

##Chapter 5-04: Feature Engineering on Databricks/Project:
* Streaming Transactions/CH5-04-Building Maximum Price Feature Table.py

In [0]:
sql("DROP TABLE IF EXISTS product_3minute_max_price_ft")
raw_transactions_df = spark.table("raw_transactions")

In [0]:
%pip install -q databricks-feature-engineering

In [0]:
# DBTITLE 1,Creating a feature table of product maximum prices.
from databricks.feature_engineering import FeatureEngineeringClient
import pyspark.sql.functions as F 

time_window = F.window(
    F.col("TransactionTimestamp"),
    windowDuration="3 minutes",
    slideDuration="1 minute",
).alias("time_window")
              
max_price_df = (
  raw_transactions_df
    .groupBy(F.col("Product"), time_window)
    .agg(F.max(F.col("Amount")).cast("float").alias("MaxProductAmount"))
    .withColumn("LookupTimestamp", 
                F.date_trunc('minute',
                          F.col("time_window.end") + F.expr('INTERVAL 1 MINUTE')))
    .drop("time_window")
)

# Create feature table with Product as the primary key
# We're using the convention of appending feature table names with "_ft"
fe = FeatureEngineeringClient()
fe.create_table(
  df=max_price_df,
  name='product_3minute_max_price_ft',
  primary_keys=['Product', 'LookupTimestamp'],
  timeseries_columns='LookupTimestamp',
  schema=max_price_df.schema,
  description="Maximum price per product over the last 3 minutes for Synthetic Transactions. Join on TransactionTimestamp to get the max product price from last minute's 3 minute rolling max"
)